# Uso básico da ferramenta Browser com Amazon Nova Act SDK

## Visão Geral

Neste tutorial, aprenderemos como usar o SDK Nova Act com a ferramenta Browser do Amazon Bedrock Agentcore. Forneceremos exemplos de uso da ferramenta de navegador sem interface (headless) e com visualização ao vivo.


### Detalhes do Tutorial


| Informação          | Detalhes                                                                          |
|:--------------------|:----------------------------------------------------------------------------------|
| Tipo de tutorial    | Conversacional                                                                    |
| Tipo de agente      | Único                                                                             |
| Framework agêntico  | Nova Act                                                                          |
| Modelo LLM          | Modelo Amazon Nova Act                                                            |
| Componentes         | Usando NovaAct para interagir com a ferramenta de navegador de forma headless    |
| Vertical do tutorial| Vertical                                                                          |
| Complexidade        | Fácil                                                                             |
| SDK utilizado       | Amazon BedrockAgentCore Python SDK, Nova Act                                      |

### Arquitetura do Tutorial

Neste tutorial, descreveremos como usar o Nova Act com a ferramenta de navegador.  

Em nosso exemplo, enviaremos instruções em linguagem natural para o agente Nova Act executar tarefas no navegador Bedrock Agentcore de forma headless.

### Principais Recursos do Tutorial

* Usando a ferramenta de navegador de forma headless
* Usando Nova Act com a ferramenta de navegador

## Pré-requisitos

Para executar este tutorial, você precisará de:
* Python 3.10+
* Credenciais AWS. Sua função/usuário IAM deve ter estas permissões https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/browser-onboarding.html#browser-credentials-config
* Amazon Bedrock AgentCore SDK
* Nova Act SDK e chave de API

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Usando NovaAct com a ferramenta Browser do Bedrock Agentcore
Criaremos um script Python que pode ser executado localmente. No script, o Nova Act usará o endpoint CDP da sessão do navegador para se conectar a ele e executar ações do Playwright.

In [ ]:
%%writefile basic_browser_with_nova_act.py
"""Script de automação de navegador usando Amazon Bedrock AgentCore e Nova Act.

Este script demonstra automação web alimentada por IA através de:
- Inicialização de uma sessão de navegador pelo Amazon Bedrock AgentCore
- Conexão ao Nova Act para interações web em linguagem natural
- Realização de buscas automatizadas e extração de dados usando o navegador
"""

from bedrock_agentcore.tools.browser_client import browser_session
from nova_act import NovaAct
from rich.console import Console
import argparse
import json

console = Console()

from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print("usando região", region)

def browser_with_nova_act(prompt, starting_page, nova_act_key, region="us-west-2"):
    result = None
    with browser_session(region) as client:
        ws_url, headers = client.generate_ws_headers()
        try:
            with NovaAct(
                cdp_endpoint_url=ws_url,
                cdp_headers=headers,
                preview={"playwright_actuation": True},
                nova_act_api_key=nova_act_key,
                starting_page=starting_page,
            ) as nova_act:
                result = nova_act.act(prompt)
        except Exception as e:
            console.print(f"Erro NovaAct: {e}")
        finally:
            return result


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--prompt", required=True, help="Instrução de busca no navegador")
    parser.add_argument("--starting-page", required=True, help="URL inicial")
    parser.add_argument("--nova-act-key", required=True, help="Chave de API Nova Act")
    parser.add_argument("--region", default="us-west-2", help="Região AWS")
    args = parser.parse_args()

    result = browser_with_nova_act(
        args.prompt, args.starting_page, args.nova_act_key, args.region
    )
    console.print(f"\n[cyan] Resposta[/cyan] {result.response}")
    console.print(f"\n[bold green]Resultado Nova Act:[/bold green] {result}")

#### Executando o script
Cole sua chave de API Nova Act abaixo antes de executar o script.

In [ ]:
NOVA_ACT_KEY= '' ### Cole sua chave Nova Act aqui

In [ ]:
!python basic_browser_with_nova_act.py --prompt "Procure por macbooks e extraia os detalhes do primeiro" --starting-page "https://www.amazon.com/" --nova-act-key {NOVA_ACT_KEY}

In [ ]:
!python basic_browser_with_nova_act.py --prompt "Extraia e retorne a receita da Amazon dos últimos 4 anos" --starting-page "https://stockanalysis.com/stocks/amzn/financials/" --nova-act-key {NOVA_ACT_KEY}

## O que aconteceu nos bastidores?

* Quando você usou `browser_session` para instanciar um cliente de navegador, ele criou um cliente Browser e iniciou uma sessão
* Em seguida, você configurou o Nova Act para apontar para aquela sessão de navegador usando `cdp_endpoint_url` e `cdp_headers` 
* O SDK Nova ACT agora pegou suas instruções em linguagem natural e criou ações do Playwright no navegador.

# Parabéns!